# ЛР3. Анализ: можно ли доверять LLM в задачах классического ML

Из этого ноутбука берётся **каждое число отчёта** (шаг 9 хода работы). Ноутбук только читает файлы:
`results/` (ответы, автопроверка, запуски кода D, тип E), `grading/` (листы двух проверяющих и ключ),
`logs/calls.jsonl` (память локальной модели), `config/` (модели, тарифы, настройки). Все числа экспортируются
в `results/summary_metrics.json`, рисунки — в `report/figures/`.

Логика подсчёта вынесена в `src/analysis.py` (там же правила: что считается «верно», как учитываются отказы
и расхождения проверяющих); функции покрыты тестами `tests/test_analysis.py`. Здесь — вызовы и комментарии.

**Выводы пишет человек** — в ячейках «Вывод (заполняет человек)». Агент выводов не пишет.

Запуск: Kernel → Restart & Run All, или
`.venv/bin/jupyter nbconvert --to notebook --execute --inplace analysis.ipynb`.

`DEMO = True` — проверка ноутбука на mock-моделях и случайных оценках из `demo/` (чисел для отчёта там нет;
на каждом рисунке и таблице пометка DEMO, файлы пишутся в `demo/`).

In [ ]:
DEMO = False   # True — демо-данные из demo/ (mock-модели, случайные оценки). Для отчёта всегда False.

## 0. Настройки и загрузка

In [ ]:
import json
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import analysis as an, autograde, bank, checks, llm, run_bank, typeE

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

P = an.paths(DEMO)                      # пути: results/, grading/, журнал, рисунки, summary
CFG = run_bank.load_experiment()        # config/experiment.json
SEED = CFG["seed"]                      # 42: бутстреп и всё случайное
N_BOOT = CFG["bootstrap_n"]             # 1000 ресэмплов задач
LANG = CFG["languages"][0]              # язык основного прогона (ru)
SELECTED = CFG["hypotheses"]            # выбранные гипотезы (закоммичены до прогонов)
MODELS = llm.load_models()              # config/models.json: роль модели → цвет и группа (H4, H5)
PRICING = an.load_pricing(P["pricing"]) # config/pricing.json
TAG = an.demo_label(DEMO)               # пометка DEMO в заголовках рисунков

summary = {"demo": DEMO, "inputs": an.input_fingerprints(P), "hypotheses_selected": SELECTED,
           "seed": SEED, "bootstrap_n": N_BOOT}
if DEMO:
    print("=" * 70 + "\nDEMO: mock-модели и случайные оценки. Это НЕ результаты эксперимента.\n" + "=" * 70)
print("Выбранные гипотезы:", SELECTED or "ещё не выбраны (config/experiment.json)")
print("Входные файлы:", *summary["inputs"], sep="\n  ")

In [ ]:
# Методика: состав банка (data/tasks.jsonl) и вызовы моделей (журнал) — отсюда эти числа берёт отчёт.
summary["bank"] = an.bank_summary(bank.load_tasks(bank.BANK_PATH))
summary["runs"] = an.runs_summary(P["log"], MODELS)
print(f"Банк: {summary['bank']['n_tasks']} задач, своих {summary['bank']['n_own']}; статусы эталонов: {summary['bank']['status']}")
an.show(summary["bank"]["tables"]["тип × тема"], "Банк: тип × тема", DEMO, floatfmt="{:.0f}")
an.show(pd.DataFrame([{"experiment": k, **v} for k, v in summary["runs"]["calls_by_experiment"].items()]),
        "Вызовы по экспериментам (журнал)", DEMO, floatfmt="{:.0f}")
an.show(pd.DataFrame([{"model_key": k, **v} for k, v in summary["runs"]["models"].items()]),
        "Модели: идентификаторы, даты, параметры (без smoke и dev)", DEMO)

### 0.1. Ручные оценки: два проверяющих, деанонимизация по ключу

Листы `sheet_r1.csv` и `sheet_r2.csv` соединяются с ключом `key_DO_NOT_OPEN.csv` по `item_id` — так у каждой
оценки появляются модель, задача и `call_id`. Итоговая оценка = **среднее двух проверяющих**
(для «верно / неверно» получается 0, 0,5 или 1). Ниже — список всех расхождений.

In [ ]:
manual = an.load_manual(P["grading"])
if manual.empty:
    print("Листов ручной проверки нет: C и D пока без оценки (G2 в HUMAN_TODO.md).")
else:
    n_incomplete = int((~manual["complete"]).sum())
    print(f"Строк в листах: {len(manual)}; заполнены обоими полностью: {len(manual) - n_incomplete}")
    if n_incomplete:
        print(f"  ВНИМАНИЕ: {n_incomplete} строк заполнены не до конца — такие ответы C/D не входят в точность")
disagree = an.disagreements(manual) if not manual.empty else pd.DataFrame()
an.show(disagree, f"Расхождения проверяющих (всего {len(disagree)})", DEMO)

### 0.2. Ответы основного прогона

Одна строка на ответ модели (`results/main_<lang>.jsonl`). `correct`: A, F, вердикт B — автопроверка
(`results/graded_auto.jsonl`, ответ разобран заново и сравнён с текущим эталоном); C — доля проверяющих,
поставивших балл ≥ 1; D — доля проверяющих, отметивших «итоговая метрика верна» (если автопроверка уже нашла
метрику в допуске, то 1). Ответы с технической ошибкой (сеть, 429, 5xx) не являются ответами модели и не входят.

In [ ]:
graded = an.read_table(P["results"], "graded_auto")
main, n_tech = an.answers(P["results"], "main", LANG, manual, graded)
HAVE_MAIN = not main.empty
if HAVE_MAIN:
    ORDER = an.order_models(main["model_key"], MODELS)      # порядок моделей: ru1, ru2, big, local
    COLORS = [an.model_color(m, MODELS) for m in ORDER]
    ungraded = main[main["correct"].isna()]
    print(f"Ответов: {len(main)} (моделей {len(ORDER)}, задач {main['task_id'].nunique()}); "
          f"техошибок (не входят): {n_tech}")
    print(f"Без оценки (в точность не входят): C/D без ручной оценки — {int(ungraded['type'].isin(['C', 'D']).sum())}, "
          f"A/B/F без автопроверки (нет эталона) — {int((~ungraded['type'].isin(['C', 'D'])).sum())}")
    print("Модели:", ", ".join(f"{m} ({MODELS.get(m, {}).get('role')})" for m in ORDER))
    print("model_reported:", {m: sorted(map(str, g.unique())) for m, g in main.groupby("model_key")["model_reported"]})
else:
    ORDER, COLORS = [], []
    print(f"Нет {run_bank.show(P['results'] / f'main_{LANG}.jsonl')}: основной прогон ещё не сделан (шаг 08).")
summary["counts"] = {"main_answers": len(main), "main_tech_errors": n_tech,
                     "main_ungraded": int(main["correct"].isna().sum()) if HAVE_MAIN else 0,
                     "models": ORDER, "manual_rows": len(manual), "disagreements": len(disagree)}

### 0.3. Выборочная проверка парсера (20 ответов, `grading/parser_spotcheck.csv`)

Доля «да» (человек согласен с разбором и автооценкой) отдельно для случайных и для «подозрительных» ответов.

In [ ]:
spot_path = P["grading"] / autograde.SPOTCHECK_NAME
spot = pd.DataFrame(autograde.read_csv(spot_path)) if spot_path.exists() else pd.DataFrame()
if not spot.empty:
    spot["agree"] = spot["human_agrees"].map(checks.norm)
    spot_table = (spot[spot["agree"].isin(["да", "нет"])].groupby("selection")["agree"]
                  .agg(n="size", n_yes=lambda s: int((s == "да").sum())).reset_index())
    spot_table["share_yes"] = spot_table["n_yes"] / spot_table["n"]
    spot_table[["ci_low", "ci_high"]] = [an.clopper_pearson(k, n) for k, n in zip(spot_table["n_yes"], spot_table["n"])]
else:
    spot_table = pd.DataFrame()
an.show(spot_table, "Согласие человека с автопарсером (CI Клоппера–Пирсона)", DEMO)
summary["parser_spotcheck"] = spot_table

## 1. Согласие проверяющих: каппа Коэна

`sklearn.metrics.cohen_kappa_score` по клеткам, заполненным обоими: B и C — балл 0–2, D — все пункты чек-листа
вместе (1 / 0 / н/п) и по отдельности, корректность метрики D (где её ставили люди). Если κ < 0,6 —
уточнить рубрику и переоценить (задание).

In [ ]:
kappa = an.kappa_table(manual)
an.show(kappa, "Каппа Коэна", DEMO)
low = kappa[kappa["below_min"]] if not kappa.empty else kappa
if len(low):
    print("\n" + "!" * 70)
    print(f"ВНИМАНИЕ: κ < {an.KAPPA_MIN} — " + "; ".join(f"{r.type}: {r.what} (κ = {r.kappa:.2f})" for r in low.itertuples()))
    print("Нужно уточнить рубрику и переоценить (HUMAN_TODO, G2). Числа C/D ниже пока ненадёжны.")
    print("!" * 70)
summary["kappa"] = kappa
summary["kappa_below_min"] = low["type"].tolist() if len(low) else []

## 2. Точность по моделям, типам и «модель × тип»; отказы

- `accuracy` — доля верных среди всех ответов: отказ («нельзя определить», отказ отвечать) считается ошибкой,
  кроме задач с недостающими данными, где это правильный ответ;
- `accuracy_answered` — доля верных только среди ответов без такого отказа (вопрос самоконтроля № 9:
  два способа посчитать точность при отказах);
- `refusal_rate` — доля ответов «нельзя определить» или отказов (все задачи);
- `ci_low`, `ci_high` — 95% CI бутстрепом: 1000 раз выбираем задачи с возвращением (seed из config).

In [ ]:
if HAVE_MAIN:
    acc_model = an.acc_table(main, ["model_key"], N_BOOT, SEED).set_index("model_key").loc[ORDER].reset_index()
    acc_type = an.acc_table(main, ["type"], N_BOOT, SEED).sort_values("type")   # CI: бутстреп задач, все модели
    acc_mt = an.acc_table(main, ["model_key", "type"], N_BOOT, SEED)
else:
    acc_model = acc_type = acc_mt = pd.DataFrame()
an.show(acc_model, "Точность по моделям (весь банк, RU)", DEMO)
an.show(acc_type, "Точность по типам задач (все модели вместе)", DEMO)
an.show(acc_mt, "Точность «модель × тип»", DEMO)
summary["accuracy_by_model"] = acc_model
summary["accuracy_by_type"] = acc_type
summary["accuracy_by_model_type"] = acc_mt

Баллы 0–2 по рубрике (B — вердикт с обоснованием, C — выбор метода): среднее двух проверяющих.

In [ ]:
if HAVE_MAIN and main["score_0_2"].notna().any():
    scores = (main[main["score_0_2"].notna()].groupby(["model_key", "type"])["score_0_2"]
              .agg(n="size", mean_score="mean").reset_index())
else:
    scores = pd.DataFrame()
an.show(scores, "Средний балл 0–2 по рубрике", DEMO)
summary["rubric_scores"] = scores

### 2.1. Тепловая карта «модель × тип задачи» (точность)

In [ ]:
if HAVE_MAIN:
    heat = acc_mt.pivot(index="model_key", columns="type", values="accuracy").reindex(index=ORDER, columns=an.BANK_TYPES)
    heat_n = acc_mt.pivot(index="model_key", columns="type", values="n_graded").reindex(index=ORDER, columns=an.BANK_TYPES)
    fig, ax = plt.subplots(figsize=(7, 0.7 * len(ORDER) + 1.5))
    im = ax.imshow(heat.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v, n = heat.iat[i, j], heat_n.iat[i, j]
            if pd.notna(v):
                ax.text(j, i, f"{v:.2f}\nn={int(n)}", ha="center", va="center", fontsize=9,
                        color="white" if v > 0.6 else "#0b0b0b")
    ax.set_xticks(range(len(an.BANK_TYPES)), an.BANK_TYPES)
    ax.set_yticks(range(len(ORDER)), ORDER)
    ax.set_xlabel("тип задачи")
    ax.set_title(TAG + "Точность «модель × тип задачи»")
    fig.colorbar(im, ax=ax, label="точность")
    an.save_fig(fig, "heatmap_model_type", P["figures"], DEMO)
    plt.show()

### 2.2. Точность моделей с 95% доверительными интервалами

In [ ]:
if HAVE_MAIN:
    fig = an.plot_share_ci(acc_model, "model_key", "accuracy", "ci_low", "ci_high",
                           TAG + "Точность по моделям, 95% CI (бутстреп задач)", "точность", colors=COLORS)
    fig.axes[0].set_xlim(0, 1)
    an.save_fig(fig, "accuracy_ci", P["figures"], DEMO)
    plt.show()

## 3. Значимость различий моделей: тест Макнемара

Все 6 пар моделей на одних и тех же задачах, `statsmodels ... mcnemar(exact=True)`: тест смотрит только на
задачи, где модели разошлись (`only_first` — верно только у первой, `only_second` — только у второй).
Задачи, где у модели C/D оценка проверяющих разошлась (0,5) или её нет, в тест не входят.
`p_holm` — p-value с поправкой Холма на 6 сравнений.

In [ ]:
mcn = an.mcnemar_pairs(main, ORDER) if HAVE_MAIN else pd.DataFrame()
an.show(mcn, "Макнемар для пар моделей (+ поправка Холма)", DEMO, floatfmt="{:.4f}")
summary["mcnemar_models"] = mcn

## 4. Тип D: код scikit-learn

- `pass_at_1` — доля программ, выполнившихся без ошибки с первого запуска (без правок, `results/d_exec.jsonl`);
- `correct` — итоговая метрика в пределах допуска от эталона (проверяющие; авто «верно» = 1);
- `method_error_any` — доля программ с ≥ 1 методологической ошибкой по чек-листу (`data/checklist_D.md`);
- `d1…d6` — доля программ с ошибкой по каждому пункту. Везде среднее двух проверяющих;
  «н/п» и ответы без кода не входят в знаменатель.

In [ ]:
typeD = an.typeD_table(main, N_BOOT, SEED) if HAVE_MAIN else pd.DataFrame()
an.show(typeD, "Тип D: запускаемость, корректность, методологические ошибки (95% CI — бутстреп задач)", DEMO)
summary["typeD"] = typeD
if not typeD.empty:
    allm = typeD[typeD["model_key"] == "все модели"].iloc[0]
    items = pd.Series({name: allm[c] for c, name in an.blind.CHECKLIST.items()})
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.barh(items.index[::-1], items.to_numpy()[::-1], color="#2a78d6")
    for y, v in enumerate(items.to_numpy()[::-1]):
        if pd.notna(v):
            ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=8, color="#52514e")
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("доля программ с ошибкой (все модели)")
    ax.set_title(TAG + "Тип D: методологические ошибки по пунктам чек-листа")
    an.save_fig(fig, "typeD_checklist", P["figures"], DEMO)
    plt.show()

## 5. Тип E: LLM как классификатор (AI4I 2020)

F1 по редкому классу «отказ» и balanced accuracy в зависимости от числа примеров в промпте (0 / 5 / 20).
LLM — по 200 строкам теста (`results/typeE_llm.jsonl`; неответ считается ошибкой, метрики только по ответам —
`*_answered`). LogReg и GaussianNB обучены на **тех же** 5 и 20 примерах (`fixed`), разброс — по 20 случайным
стратифицированным наборам (`resample`, среднее ± sd). Константа «всегда нет отказа»: на тесте и на всём датасете
(accuracy при естественной доле отказов). Таблица та же, что пишет `src.typeE_metrics`.

In [ ]:
typeE_tab = an.typeE_table(P["results"])
cols = ["source", "model", "shots", "train_set", "eval_set", "n_eval", "coverage", "f1_failure",
        "balanced_accuracy", "accuracy", "f1_failure_sd", "balanced_accuracy_sd", "f1_failure_answered",
        "balanced_accuracy_answered", "complete"]
an.show(typeE_tab[cols], "Тип E: метрики LLM и классических моделей", DEMO)
summary["typeE"] = typeE_tab

In [ ]:
# 95% CI для F1 «отказа» и balanced accuracy: бутстреп 200 строк теста (1000 раз строки с возвращением).
# LLM — неответ считается ошибкой (как в таблице выше); LogReg — обучена на тех же 5 и 20 примерах, что видит LLM.
splitE = typeE.load_split()
predsE_ci = an.typeE_llm_preds(P["results"], splitE, LANG)
if predsE_ci.empty:
    typeE_ci = pd.DataFrame()
    print("Нет ответов LLM типа E на текущем тесте (results/typeE_llm.jsonl).")
else:
    lrE = an.logreg_rows(an.logreg_predictions(splitE, SEED), predsE_ci)
    typeE_ci = an.typeE_ci_table(pd.concat([predsE_ci, lrE]), N_BOOT, SEED).sort_values(["shots", "model_key"])
an.show(typeE_ci, "Тип E: F1 «отказа» и balanced accuracy с 95% CI", DEMO)
summary["typeE_ci"] = typeE_ci


In [ ]:
llmE = typeE_tab[typeE_tab["source"] == "llm"]
cls = typeE_tab[typeE_tab["source"] == "classical"]
if llmE.empty:
    print("Ответов LLM типа E нет (шаг 09) — рисунок не строится.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
    for ax, metric, name in [(axes[0], "f1_failure", "F1 класса «отказ»"), (axes[1], "balanced_accuracy", "balanced accuracy")]:
        for m in an.order_models(llmE["model"], MODELS):
            g = llmE[llmE["model"] == m].sort_values("shots")
            ax.plot(g["shots"], g[metric], "o-", lw=2, ms=8, color=an.model_color(m, MODELS), label=f"LLM {m}")
        for model, style in [("LogisticRegression", "s--"), ("GaussianNB", "^:")]:
            fx = cls[(cls["model"] == model) & (cls["train_set"] == "fixed")].sort_values("shots")
            rs = cls[(cls["model"] == model) & (cls["train_set"] == "resample")].sort_values("shots")
            ax.plot(fx["shots"], fx[metric], style, color="#52514e", ms=8, label=f"{model} (те же примеры)")
            ax.errorbar(rs["shots"] + 0.6, rs[metric], yerr=rs[f"{metric}_sd"], fmt="none", ecolor="#8a8984",
                        capsize=4, label=f"{model}: 20 наборов, среднее ± sd")
        const = cls[(cls["model"] == "Dummy(most_frequent)") & (cls["eval_set"] == "test")][metric]
        if len(const):
            ax.axhline(const.iloc[0], color="#e34948", ls="--", lw=1, label="константа «нет отказа» (тест)")
        ax.set_title(name)
        ax.set_xticks([0, 5, 20])
        ax.set_xlabel("число примеров в промпте / в обучении")
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", color="#e6e5e0")
    axes[1].legend(fontsize=7, frameon=False, loc="center left", bbox_to_anchor=(1, 0.5))
    fig.suptitle(TAG + "Тип E: качество в зависимости от числа примеров (200 строк теста)")
    an.save_fig(fig, "typeE_shots", P["figures"], DEMO)
    plt.show()
full = cls[cls["eval_set"] == "full"]
if len(full):
    print(f"Константа «нет отказа» на всём датасете: accuracy = {full['accuracy'].iloc[0]:.4f} "
          f"(отказов {int(full['n_eval_failures'].iloc[0])} из {int(full['n_eval'].iloc[0])})")

## 6. Гипотезы

Ниже блок для каждой из H1–H13: метрика и тест из `docs/hypotheses_guide.md`. Блок считается, если для него есть
данные. Выбранные гипотезы помечены «ВЫБРАНА»; остальные — разведочный анализ (их можно не включать в отчёт).
Пороги решения берутся из `hypotheses.md` (зафиксированы коммитом до прогонов), здесь их нет.

In [ ]:
def hyp(h: str) -> None:
    status = "ВЫБРАНА (прогноз в hypotheses.md)" if h in SELECTED else "не выбрана — разведочный анализ"
    print(f"{TAG}{h}: {status}")


summary["hypotheses"] = {}

### H1. Учебниковые задачи против изменённых чисел
Пары `pair_id`: оригинал (A, «учебник») ↔ изменённая версия (F, `modified`). Точность на оригиналах и изменённых, разница с бутстреп-CI по парам, Макнемар по всем моделям вместе.

In [ ]:
hyp("H1")
h1 = an.h1_pairs(main) if HAVE_MAIN else pd.DataFrame()
if h1.empty:
    print("Нет пар оригинал ↔ изменённая версия с ответами (нужны заполненные F1–F5 и main).")
else:
    h1_tab = an.name_conditions(an.compare_conditions(h1, "model_key", N_BOOT, SEED, cluster="pair_id"),
                                "original", "modified")
    an.show(h1_tab, "H1: оригинал vs изменённая версия (CI разницы — бутстреп пар)", DEMO)
    fig = an.plot_two_conditions(h1_tab, "model_key", "acc_original", "acc_modified", "оригинал (учебник)",
                                 "изменённая версия", TAG + "H1: точность на парах задач")
    an.save_fig(fig, "H1_original_vs_modified", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H1"] = {"table": h1_tab}

**Вывод (заполняет человек):** … _H1: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H2. Игнорирование базовой частоты: формульная vs словесная версия
Эксперимент `verbal_formula`: одни и те же задачи в формульной и словесной записи. Точность в каждой версии с CI Клоппера–Пирсона, Макнемар по всем моделям вместе.

In [ ]:
hyp("H2")
vf, _ = an.answers(P["results"], "verbal_formula", LANG, manual, graded)
if vf.empty:
    print("Нет results/verbal_formula_<lang>.jsonl (режим verbal_formula не запускался).")
else:
    h2 = an.paired_conditions(vf[vf["variant"] == "formula"], vf[vf["variant"] == "verbal"], ["model_key", "task_id"])
    h2_tab = an.name_conditions(an.compare_conditions(h2, "model_key", N_BOOT, SEED), "formula", "verbal")
    an.show(h2_tab, "H2: формульная vs словесная версия", DEMO)
    fig = an.plot_two_conditions(h2_tab, "model_key", "acc_formula", "acc_verbal", "формульная", "словесная",
                                 TAG + "H2: точность в формульной и словесной версии")
    an.save_fig(fig, "H2_formula_vs_verbal", P["figures"], DEMO)
    plt.show()
    print("Доля ответов, близких к P(+|D) (игнорирование априорной вероятности): в банке нет поля с этим числом — "
          "при необходимости посчитать по results/verbal_formula_<lang>.jsonl вручную.")
    summary["hypotheses"]["H2"] = {"table": h2_tab}

**Вывод (заполняет человек):** … _H2: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H3. Додумывание при недостающих данных
Задачи `answer_kind = cannot_determine` (F6–F8): доля ответов-чисел («додумала») с CI Клоппера–Пирсона, доля «нельзя определить» / отказов. Контроль — доля отказов на ответимых задачах той же пары (A6–A8).

In [ ]:
hyp("H3")
h3 = an.h3_table(main) if HAVE_MAIN else pd.DataFrame()
an.show(h3, "H3: задачи с недостающими данными", DEMO)
if not h3.empty:
    fig = an.plot_share_ci(h3, "model_key", "share_invented", "ci_low", "ci_high",
                           TAG + "H3: доля «додуманных» чисел, 95% CI", "доля ответов-чисел", ref=0.5)
    fig.axes[0].set_xlim(0, 1)
    an.save_fig(fig, "H3_invented", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H3"] = {"table": h3}

**Вывод (заполняет человек):** … _H3: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H4. Язык запроса: RU vs EN
По каждой модели точность RU и EN, Δ = RU − EN и Макнемар; разница средних Δ российских (роли ru1, ru2) и зарубежных моделей с бутстреп-CI по задачам.

In [ ]:
hyp("H4")
en, _ = an.answers(P["results"], "main", "en", manual, graded)
if en.empty or not HAVE_MAIN:
    print("Нет results/main_en.jsonl (режим lang_en не запускался).")
else:
    h4_tab, h4_group = an.h4_table(main, en, MODELS, N_BOOT, SEED)
    h4_tab = an.name_conditions(h4_tab, "ru", "en")
    an.show(h4_tab, "H4: RU vs EN по моделям", DEMO)
    print("Разница Δ (российские − зарубежные):", an.rounded(h4_group))
    fig = an.plot_two_conditions(h4_tab, "model_key", "acc_ru", "acc_en", "RU", "EN", TAG + "H4: точность RU и EN")
    an.save_fig(fig, "H4_ru_vs_en", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H4"] = {"table": h4_tab, "group_difference": h4_group}

**Вывод (заполняет человек):** … _H4: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H5. Маленькая модель: A vs B
Отставание локальной модели от средней точности больших на A и на B, разница отставаний (A − B) с бутстреп-CI, Макнемар локальной против каждой большой.

In [ ]:
hyp("H5")
h5_tab, h5_sum = an.h5_table(main, MODELS, N_BOOT, SEED) if HAVE_MAIN else (pd.DataFrame(), {})
an.show(h5_tab, "H5: точность больших моделей и локальной на A и B (value — точность большой модели)", DEMO)
if h5_sum:
    print("Разница отставаний:", an.rounded(h5_sum))
    gaps = pd.DataFrame({"type": ["A", "B"], "gap": [h5_sum["gap_A"], h5_sum["gap_B"]]})
    fig, ax = plt.subplots(figsize=(4.5, 3.2))
    ax.bar(gaps["type"], gaps["gap"], color="#2a78d6", width=0.5)
    ax.axhline(0, color="#8a8984", lw=1)
    ax.set_ylabel("отставание (большие − локальная)")
    ax.set_title(TAG + "H5: отставание локальной модели")
    an.save_fig(fig, "H5_gap", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H5"] = {"table": h5_tab, "summary": h5_sum}

**Вывод (заполняет человек):** … _H5: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H6. «Думай пошагово»
Точность main и cot на A и на B (Макнемар по всем моделям вместе), отношение токенов, стоимости и задержки cot / main. Модель режима рассуждений (`qwen3-8b-think`, если запускалась) сравнивается с `qwen3-8b` из main **отдельной группой**: это другое воздействие, а ответы main у них общие.

In [ ]:
hyp("H6")
cot, _ = an.answers(P["results"], "cot", LANG, manual, graded)
if cot.empty or not HAVE_MAIN:
    print("Нет results/cot_<lang>.jsonl (режим cot не запускался).")
else:
    h6_acc, h6_tok = an.h6_tables(main, cot, MODELS, PRICING, N_BOOT, SEED)
    an.show(h6_acc, "H6: точность cot vs main по типам (все модели вместе; режим рассуждений — отдельно)", DEMO)
    an.show(h6_tok, "H6: отношение токенов, стоимости и задержки cot / main", DEMO)
    fig = an.plot_two_conditions(h6_acc, "group", "acc_main", "acc_cot", "обычный промпт (main)", "CoT / рассуждения",
                                 TAG + "H6: точность без CoT и с CoT")
    an.save_fig(fig, "H6_cot", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H6"] = {"accuracy": h6_acc, "tokens": h6_tok}

**Вывод (заполняет человек):** … _H6: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H7. Самосогласованность: 5 ответов при T = 0,7 и голосование
Точность ответа большинства (`selfconsistency_vote`) против main; Макнемар. Разброс — доля задач, где повторы дали не один и тот же ответ (больше одной группы ответов или часть ответов не разобрана), с CI.

In [ ]:
hyp("H7")
votes = an.read_table(P["results"], f"selfconsistency_vote_{LANG}")
if votes.empty or not HAVE_MAIN:
    print("Нет results/selfconsistency_vote_<lang>.jsonl (режим selfconsistency не запускался).")
else:
    repeats = an.read_table(P["results"], f"selfconsistency_{LANG}")     # все 5 повторов (для техошибок)
    h7_acc, h7_spread = an.h7_tables(main, votes, repeats, graded, N_BOOT, SEED)
    an.show(h7_acc, "H7: большинство из 5 vs один ответ main", DEMO)
    an.show(h7_spread, "H7: разброс между повторами", DEMO)
    fig = an.plot_two_conditions(h7_acc, "model_key", "acc_main", "acc_vote", "main (T = 0)", "большинство из 5 (T = 0,7)",
                                 TAG + "H7: точность одного ответа и голосования")
    an.save_fig(fig, "H7_selfconsistency", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H7"] = {"accuracy": h7_acc, "spread": h7_spread}

**Вывод (заполняет человек):** … _H7: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H8. Устойчивость к перефразированию
Доля пар «исходная формулировка / перефразировка», где правильность ответа не совпала (1 − устойчивость), с CI Клоппера–Пирсона.

In [ ]:
hyp("H8")
para, _ = an.answers(P["results"], "paraphrase", LANG, manual, graded)
if para.empty or not HAVE_MAIN:
    print("Нет results/paraphrase_<lang>.jsonl (режим paraphrase не запускался).")
else:
    h8 = an.mismatch_table(an.paired_conditions(main, para, ["model_key", "task_id"]))
    an.show(h8, "H8: доля пар с разной правильностью", DEMO)
    fig = an.plot_share_ci(h8, "model_key", "share_mismatch", "ci_low", "ci_high",
                           TAG + "H8: доля пар с разной правильностью, 95% CI", "доля пар", ref=0.1)
    fig.axes[0].set_xlim(0, 1)
    an.save_fig(fig, "H8_paraphrase", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H8"] = {"table": h8}

**Вывод (заполняет человек):** … _H8: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H9. Поддакивание
Из ответов, верных в main, доля ставших неверными после «Мне кажется, ты ошибся» (flip rate), с CI Клоппера–Пирсона. Отдельно — сколько вторых ответов не разобрано или с отказом.

In [ ]:
hyp("H9")
follow, _ = an.answers(P["results"], "followup_doubt", LANG, manual, graded)
if follow.empty or not HAVE_MAIN:
    print("Нет results/followup_doubt_<lang>.jsonl (режим followup_doubt не запускался).")
else:
    h9 = an.h9_table(main, follow)
    an.show(h9, "H9: flip rate", DEMO)
    fig = an.plot_share_ci(h9, "model_key", "flip_rate", "ci_low", "ci_high",
                           TAG + "H9: доля «переворотов», 95% CI", "flip rate", ref=0.1)
    fig.axes[0].set_xlim(0, 1)
    an.save_fig(fig, "H9_flip_rate", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H9"] = {"table": h9}

**Вывод (заполняет человек):** … _H9: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H10. Калибровка уверенности (ECE, диаграмма надёжности)
Ответы A, B, F основного прогона с указанной уверенностью и бинарной автопроверкой. Ответы делятся на 5 корзин
равной ширины по уверенности (0–20, …, 80–100); ECE — взвешенное среднее |уверенность − доля верных|.
«Уверенность − точность» и ECE — с бутстреп-CI. Пропуски уверенности не заполняются (`n_no_confidence`).

In [ ]:
hyp("H10")
calib_src = main[main["type"].isin(an.CALIB_TYPES)] if HAVE_MAIN else pd.DataFrame()
if calib_src.empty:
    print("Нет ответов A, B, F в main.")
else:
    h10 = an.calibration_table(calib_src, N_BOOT, SEED).set_index("model_key").loc[ORDER].reset_index()
    an.show(h10, "H10: уверенность, точность, ECE", DEMO)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot([0, 1], [0, 1], color="#8a8984", ls="--", lw=1, label="идеальная калибровка")
    bins = {}
    for m, c in zip(ORDER, COLORS):
        g = calib_src[(calib_src["model_key"] == m) & calib_src["confidence"].notna() & calib_src["correct"].apply(an.binary)]
        _, tb = an.ece(g["confidence"].to_numpy(float), g["correct"].to_numpy(float))
        bins[m] = tb
        tb = tb[tb["n"] > 0]
        ax.plot(tb["mean_confidence"], tb["accuracy"], "o-", color=c, lw=2, ms=8, label=m)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("средняя заявленная уверенность в корзине")
    ax.set_ylabel("доля верных в корзине")
    ax.set_title(TAG + "H10: диаграмма надёжности (5 корзин)")
    ax.legend(frameon=False, fontsize=8)
    an.save_fig(fig, "H10_reliability", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H10"] = {"table": h10, "bins": bins}

**Вывод (заполняет человек):** … _H10: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H11. Код запускается, но методологически неверен
По программам D: доля незапускающихся (1 − pass@1) и доля с ≥ 1 методологической ошибкой (проверяющие); Макнемар по программам (без расхождений проверяющих).

In [ ]:
hyp("H11")
h11_rows, h11 = an.h11_table(main) if HAVE_MAIN else (pd.DataFrame(), {})
if not h11:
    print("Нет ответов D в main.")
else:
    print(an.rounded(h11))
    fig, ax = plt.subplots(figsize=(5, 3.2))
    vals = [h11["share_not_run"], h11["share_method_error"]]
    ax.bar(["не запустилась", "≥1 методологическая\nошибка"], vals, color=["#eb6834", "#2a78d6"], width=0.5)
    for i, v in enumerate(vals):
        if pd.notna(v):
            ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9, color="#52514e")
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("доля программ D")
    ax.set_title(TAG + "H11: ошибки запуска и методологические ошибки")
    an.save_fig(fig, "H11_code_errors", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H11"] = h11

**Вывод (заполняет человек):** … _H11: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

### H12. Few-shot LLM против логистической регрессии
Те же 5 и 20 примеров и те же 200 строк теста. LogReg переобучается здесь на примерах из `data/e_split.json` (как в `src.typeE classical`). Разница balanced accuracy и F1 (LLM − LogReg) с бутстреп-CI по строкам теста; Макнемар по правильности строк. Неответ LLM = ошибка.

In [ ]:
hyp("H12")
split = typeE.load_split()
predsE = an.typeE_llm_preds(P["results"], split, LANG)
if predsE.empty:
    print("Нет ответов LLM типа E на текущем тесте (results/typeE_llm.jsonl).")
else:
    h12 = an.h12_table(predsE, an.logreg_predictions(split, SEED), N_BOOT, SEED)
    h12["model_key"] = pd.Categorical(h12["model_key"], an.order_models(h12["model_key"], MODELS))
    h12 = h12.sort_values(["shots", "model_key"]).reset_index(drop=True)
    an.show(h12, "H12: LLM − LogReg на тех же примерах", DEMO, floatfmt="{:.4f}")
    h12["label"] = h12["model_key"].astype(str) + ", " + h12["shots"].astype(str) + " прим."
    fig = an.plot_share_ci(h12, "label", "ba_diff", "ba_diff_ci_low", "ba_diff_ci_high",
                           TAG + "H12: balanced accuracy LLM − LogReg, 95% CI", "разница balanced accuracy",
                           colors=[an.model_color(str(m), MODELS) for m in h12["model_key"]], ref=0)
    an.save_fig(fig, "H12_llm_vs_logreg", P["figures"], DEMO)
    plt.show()
    summary["hypotheses"]["H12"] = {"table": h12.drop(columns="label")}

**Вывод (заполняет человек):** … _H12: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

## 7. Цена и скорость

По основному прогону: средняя и медианная задержка, токены на задачу, стоимость по `config/pricing.json`
(₽ = токены_вх · цена_вх / 10⁶ + токены_вых · цена_вых / 10⁶; по официальным тарифам, даже если использовался
бесплатный лимит), ₽ за правильный ответ с бутстреп-CI. Для локальной модели цена 0 ₽ — вместо неё время и память.

In [ ]:
if not PRICING.get("verified_by_human"):
    print("ВНИМАНИЕ: тарифы в config/pricing.json ещё не сверены человеком (G2) — ₽ предварительные.")
cost = an.cost_table(main, PRICING, MODELS, N_BOOT, SEED).set_index("model_key").loc[ORDER].reset_index() \
    if HAVE_MAIN else pd.DataFrame()
an.show(cost, "Цена и скорость по моделям (основной прогон)", DEMO, floatfmt="{:.4f}")
summary["cost"] = cost
summary["pricing"] = {"verified_by_human": PRICING.get("verified_by_human"), "date_checked": PRICING.get("date_checked"),
                      "usd_rub": PRICING.get("usd_rub", {}).get("rate")}

# Тип E — основной расход токенов: суммарная стоимость по моделям (все число примеров вместе).
if len(llmE):
    e_cost = llmE.groupby("model")[["tokens_in_sum", "tokens_out_sum"]].sum().reset_index()
    prices = [an.price_for(m, PRICING, MODELS) for m in e_cost["model"]]
    e_cost["cost_rub_total"] = [np.nan if p is None else ti * p[0] / 1e6 + to * p[1] / 1e6
                                for p, ti, to in zip(prices, e_cost["tokens_in_sum"], e_cost["tokens_out_sum"])]
else:
    e_cost = pd.DataFrame()
an.show(e_cost, "Тип E: токены и стоимость всего прогона", DEMO, floatfmt="{:.2f}")
summary["cost_typeE"] = e_cost

Локальная модель: время и память на нашем железе (память — из журнала, `/api/ps` Ollama; железо — `run_manifest.json`).

In [ ]:
mem = an.local_memory(P["log"], set(main["call_id"])) if HAVE_MAIN else pd.DataFrame()
an.show(mem, "Память локальной модели (МБ)", DEMO, floatfmt="{:.0f}")
manifest = json.loads(P["manifest"].read_text(encoding="utf-8")) if P["manifest"].exists() else {}
server = manifest.get("ollama_server", {})
if server:
    print("Сервер локальной модели (run_manifest.json → ollama_server):")
    print(json.dumps(server, ensure_ascii=False, indent=1)[:1500])
summary["local_model"] = {"memory": mem, "server": server}

### 7.1. «Стоимость – точность»

In [ ]:
if HAVE_MAIN:
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for r, c in zip(cost.itertuples(), COLORS):
        a = acc_model.set_index("model_key").loc[r.model_key]
        x = 0 if r.local else r.cost_rub_per_task
        ax.errorbar(x, a["accuracy"], yerr=[[a["accuracy"] - a["ci_low"]], [a["ci_high"] - a["accuracy"]]],
                    fmt="o", color=c, ms=9, capsize=4, lw=2)
        label = r.model_key + (f"\n(локальная: 0 ₽, {r.latency_s_mean:.1f} с/задачу)" if r.local else "")
        ax.annotate(label, (x, a["accuracy"]), textcoords="offset points", xytext=(8, 4), fontsize=8, color="#0b0b0b")
    ax.set_xlabel("₽ на одну задачу (по официальным тарифам)")
    ax.set_ylabel("точность (95% CI)")
    ax.set_ylim(0, 1.05)
    ax.set_title(TAG + "Стоимость – точность (основной прогон)")
    ax.grid(color="#e6e5e0")
    ax.set_axisbelow(True)
    an.save_fig(fig, "cost_accuracy", P["figures"], DEMO)
    plt.show()

### H13. Стоимость правильного ответа
Самая точная модель и лучшая по ₽ за правильный ответ (среди платных; локальная считается отдельно — время и память выше); Макнемар между ними. CI ₽ за правильный ответ — в таблице цены.

In [ ]:
hyp("H13")
h13 = an.h13_summary(cost, main) if HAVE_MAIN else {}
print(an.rounded(h13) if h13 else "Нет данных (основной прогон или цены).")
if h13:
    summary["hypotheses"]["H13"] = {"summary": h13, "cost": cost}

**Вывод (заполняет человек):** … _H13: сравнить с прогнозом из коммита гипотез, вердикт «подтвердилась / опровергнута / данных недостаточно»._

## 8. Экспорт всех чисел: `results/summary_metrics.json`

Отчёт берёт числа **только** отсюда. В демо-режиме файл пишется в `demo/results/`.

In [ ]:
if HAVE_MAIN or len(llmE):
    an.write_summary(summary, P["summary"])
    print("Записано:", run_bank.show(P["summary"]))
    print("Рисунки:", *sorted(run_bank.show(f) for f in P["figures"].glob("*.png")), sep="\n  ")
else:
    print("Нет ни основного прогона, ни ответов LLM типа E — summary_metrics.json не пишется.")